# 1. Extração e Consolidação dos Dados Brutos

In [4]:
import glob
import pandas as pd

# 1. Localiza os ficheiros CSV na pasta data/raw
arquivos = sorted(glob.glob("../data/raw/*.csv"))
print(f"Ficheiros encontrados ({len(arquivos)}): {arquivos}")

# 2. Lê e empilha os três anos num único DataFrame
dfs = [pd.read_csv(arq, sep=";", encoding="latin1", low_memory=False) for arq in arquivos]
df_raw = pd.concat(dfs, ignore_index=True)

# 3. Diagnóstico inicial
print(f"\nDimensões totais: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
display(df_raw.head(3))
print("\n--- Tipos de dados e nulos ---")
df_raw.info()

Ficheiros encontrados (3): ['../data/raw\\datatran2024.csv', '../data/raw\\datatran2025.csv', '../data/raw\\datatran2026.csv']

Dimensões totais: 200,626 linhas x 30 colunas


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,571789.0,2024-01-01,segunda-feira,03:56:00,ES,101,38,CONCEICAO DA BARRA,Ultrapassagem Indevida,Colisão lateral sentido oposto,...,1,1,1,1,3,"-18,48261","-39,92379",SPRF-ES,DEL04-ES,UOP02-DEL04-ES
1,571804.0,2024-01-01,segunda-feira,04:50:00,PI,343,185,PIRIPIRI,Manobra de mudança de faixa,Colisão frontal,...,0,1,0,0,2,"-4,29603281","-41,76732659",SPRF-PI,DEL02-PI,UOP01-DEL02-PI
2,571806.0,2024-01-01,segunda-feira,04:30:00,BA,116,578,BREJOES,Ingestão de álcool pelo condutor,Colisão frontal,...,0,1,2,0,4,"-13,07158302","-39,9611107",SPRF-BA,DEL03-BA,UOP02-DEL03-BA



--- Tipos de dados e nulos ---
<class 'pandas.DataFrame'>
RangeIndex: 200626 entries, 0 to 200625
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      200626 non-null  float64
 1   data_inversa            200626 non-null  str    
 2   dia_semana              200626 non-null  str    
 3   horario                 200626 non-null  str    
 4   uf                      200626 non-null  str    
 5   br                      200626 non-null  int64  
 6   km                      200626 non-null  str    
 7   municipio               200626 non-null  str    
 8   causa_acidente          200626 non-null  str    
 9   tipo_acidente           200626 non-null  str    
 10  classificacao_acidente  200623 non-null  str    
 11  fase_dia                200626 non-null  str    
 12  sentido_via             200626 non-null  str    
 13  condicao_metereologica  200626 non-null  str    
 14 

# 2. Diagnóstico de Qualidade e Sujeira

In [5]:
# 1. Verifica todas as colunas e a contagem exata de valores nulos (NaN)
nulos = df_raw.isnull().sum()
print("--- Colunas com valores nulos (NaN) ---")
print(nulos[nulos > 0])

# 2. Verifica colunas categóricas para identificar valores "Ignorado", "(null)" ou erros de digitação
colunas_categoricas = [
    "dia_semana", "uf", "causa_acidente", "tipo_acidente",
    "classificacao_acidente", "fase_dia", "condicao_metereologica", "tipo_pista"
]

for col in colunas_categoricas:
    print(f"\n=== {col.upper()} (valores únicos: {df_raw[col].nunique()}) ===")
    print(df_raw[col].value_counts(dropna=False).head(8))

# 3. Verifica linhas duplicadas pelo ID da ocorrência
duplicados = df_raw.duplicated(subset=["id"]).sum()
print(f"\nRegistros com ID duplicado: {duplicados}")

--- Colunas com valores nulos (NaN) ---
classificacao_acidente      3
regional                    3
delegacia                  71
uop                       140
dtype: int64

=== DIA_SEMANA (valores únicos: 7) ===
dia_semana
domingo          31956
sábado           31911
sexta-feira      30943
segunda-feira    28025
quinta-feira     26379
quarta-feira     26062
terça-feira      25350
Name: count, dtype: int64

=== UF (valores únicos: 27) ===
uf
MG    26008
SC    22862
PR    21098
RJ    17787
RS    13552
SP    12905
BA    11451
GO     9100
Name: count, dtype: int64

=== CAUSA_ACIDENTE (valores únicos: 70) ===
causa_acidente
Ausência de reação do condutor                               30917
Reação tardia ou ineficiente do condutor                     30082
Acessar a via sem observar a presença dos outros veículos    19410
Condutor deixou de manter distância do veículo da frente     12279
Manobra de mudança de faixa                                  11322
Velocidade Incompatível             

## 3. Análise de Consistência (Datas e Numéricos)

In [6]:
# 1. Verifica o intervalo de datas e horários
print("--- Intervalo de Datas ---")
print("Mínima:", df_raw["data_inversa"].min(), "| Máxima:", df_raw["data_inversa"].max())

# 2. Testa a conversão de km, latitude e longitude (trocando vírgula por ponto)
for col in ["km", "latitude", "longitude"]:
    serie_num = pd.to_numeric(df_raw[col].astype(str).str.replace(",", "."), errors="coerce")
    print(f"{col}: nulos após conversão = {serie_num.isnull().sum()} | min = {serie_num.min()} | max = {serie_num.max()}")

# 3. Estatísticas das colunas inteiras de vítimas e veículos
cols_vitimas = ["pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "feridos", "veiculos"]
display(df_raw[cols_vitimas].describe().round(2))

# 4. Verifica se existem strings "(null)" ou "Ignorado" escondidas em colunas de texto
for col in df_raw.select_dtypes(include="object").columns:
    qtd_null_str = (df_raw[col].astype(str).str.lower().isin(["(null)", "null", "nan", "none"])).sum()
    if qtd_null_str > 0:
        print(f"Coluna '{col}' tem {qtd_null_str} valores '(null)' como texto")

--- Intervalo de Datas ---
Mínima: 2024-01-01 | Máxima: 2026-09-30
km: nulos após conversão = 0 | min = 0.0 | max = 1470.0
latitude: nulos após conversão = 0 | min = -33.68932614 | max = 4.461419
longitude: nulos após conversão = 0 | min = -72.83575058 | max = -32.406822


,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos
count,200626.00,200626.00,200626.00,200626.00,200626.00,200626.00,200626.00,200626.0
mean,2.60,0.08,0.88,0.28,1.06,0.40,1.15,2.0
std,2.25,0.36,1.08,0.62,1.80,0.84,1.20,1.1
min,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.0
25%,2.00,0.00,0.00,0.00,0.00,0.00,1.00,1.0
50%,2.00,0.00,1.00,0.00,1.00,0.00,1.00,2.0
75%,3.00,0.00,1.00,0.00,1.00,1.00,1.00,2.0
max,94.00,37.00,84.00,35.00,85.00,81.00,85.00,82.0


C:\Users\Antonio\AppData\Local\Temp\ipykernel_7276\2903079574.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_raw.select_dtypes(include="object").columns:


## 4. Resumo das Regras de Limpeza para o Pipeline ETL

- Conversão de id para inteiro (int64).

- Conversão de km, latitude e longitude de string (com vírgula) para float64 (com ponto).

- Conversão de data_inversa para datetime e extração de ano, mes, nome_mes, dia e turno (Madrugada, Manhã, Tarde, Noite) a partir de horario.

- Preenchimento dos valores ausentes em classificacao_acidente, regional, delegacia e uop com "Não informado".

- Modelagem dimensional (Star Schema) para carregamento no SQLite.